<img src="img/pandora2d_logo.png" width="500">

# Pandora2D : a coregistration framework

# Usage with origin coordinates

It presents how to use as inputs information about initial disparity.

This process will take place in two stages: first, a general framework will be established to determine the initial disparity (at the pixel level), and then the disparity maps from the first run will be used as prior information to parameterise the disparity at the sub-pixel level (local search).

It will have an additionnal step (in rows and columns) and a ROI in the image.

#### Imports and external functions

In [ ]:
import io
from pathlib import Path
from pprint import pprint

import numpy as np
from IPython.display import Image, display

In [ ]:
def plot_state_machine(pandora2d_machine):
    """
    Show the schemes of step of Pandora2D Machine
    """
    stream = io.BytesIO()
    try:
        pandora2d_machine.get_graph().draw(stream, prog='dot', format='png')
        display(Image(stream.getvalue()))
    except:
        print("It is not possible to show the graphic of the state machine. To solve it, please install graphviz on your system (apt-get install graphviz if operating in Linux) and install python package with pip install graphviz")

In [ ]:
from snippets.utils import *

## Pandora2D's pipeline

Pandora2D provides the following steps:
* estimation computation
* matching cost computation
* cost volume confidence (**first version implemented**)
* disparity computation (**mandatory if matching_cost**)
* subpixel disparity refinement

The aim is to trigger twice the following Pandora2D pipeline.
The second run use initial disparity as origin coordinates to find disparities locally at sub-pixel precision.

<img src="img/Pandora2D_pipeline.drawio.svg" width="1000">

# Pandora2D execution - Imports and input data

#### Imports of pandora2d

In [ ]:
# Load pandora2d imports
import pandora2d
from pandora2d.check_configuration import check_conf
from pandora2d.img_tools import create_datasets_from_inputs, get_roi_processing
from pandora2d.state_machine import Pandora2DMachine

#### Load and visualize input data 

Provide image path

In [ ]:
# Paths to left and right images
img_left_path = "data/left.tif"
img_right_path = "data/right.tif"

Provide output directory to write results

In [ ]:
output_dir = Path.cwd() / "output"
# If necessary, create output dir
output_dir.mkdir(exist_ok=True,parents=True)

Convert input data to dataset

In [ ]:
input_config = {
    "left": {"img": img_left_path, "nodata": np.nan},
    "right": {"img": img_right_path, "nodata": np.nan},
    "col_disparity": {"init": 0, "range": 2},
    "row_disparity": {"init": 0, "range": 2},
}

In [ ]:
img_left, img_right = create_datasets_from_inputs(input_config=input_config)

`create_datasets_from_inputs` returns a namedTuple so we could have used:

```python
image_datasets = create_datasets_from_inputs(input_config=input_config)
```

and called:

 `image_datasets.left` or `image_datasets.right` instead of `img_left` and `img_right`.

In [ ]:
fig = plt.figure(figsize=(10,10))
ax0 = fig.add_subplot(1,2,1)
ax0.imshow(img_left["im"].data)
plt.title("Left image")
ax1 = fig.add_subplot(1,2,2)
ax1.imshow(img_right["im"].data)
plt.title("Right image");

# Step 1 : Compute at pixel's level

This step will perform no sub-pixels options (''_subpix_'' key in matching_cost or refinement state) to find the origin coordinates used as initial diparity in the next step.

Step 2 will perform sub-pixel's precision using the output of this first step.

#### Instantiate the machine

In [ ]:
pandora2d_machine = Pandora2DMachine()

#### Define pipeline configuration

In [ ]:
user_cfg = {
    "input": {
        "left": {
            "img": "data/left.tif",
            "nodata": "NaN",
        },
        "right": {
            "img": "data/right.tif",
        },
        "col_disparity": {"init": 0, "range": 2},
        "row_disparity": {"init": 0, "range": 2},
    },
    "ROI":{
        "col": {"first": 10, "last": 100},
        "row": {"first": 10, "last": 100}
    },
    "pipeline":{
        "matching_cost" : {
            "matching_cost_method": "zncc",
            "window_size": 5,
            "step" : [5,3]
        },
        "disparity": {
            "disparity_method": "wta",
            "invalid_disparity": np.nan
        }
    },
    "output": {
        "path": "outputs/origin_coordinates_pixel_level_run"
    },
}

#### Check the configuration and sequence of steps

In [ ]:
checked_cfg = check_conf(user_cfg, pandora2d_machine)

In [ ]:
pipeline_cfg = checked_cfg['pipeline']
pprint(pipeline_cfg)

#### Run Pandora2D machine

In [ ]:
dataset, completed_cfg = pandora2d.run_pandora2d(pandora2d_machine,checked_cfg)

Save disparity maps

In [ ]:
pandora2d.common.save_disparity_maps(dataset, completed_cfg)

Visualize output disparity map

In [ ]:
plot_two_images(dataset["row_map"].data,
                dataset["col_map"].data,
                "Row disparity map",
                "Columns disparity map", 
                output_dir, 
                cmap=pandora_cmap())

With associated validity masks (detailed in the readthedocs).

Valid points are flagged as ''0''.

In [ ]:
plot_two_images(dataset["validity"].data[..., 0],
                dataset["validity"].data[..., 1],
                "Validity map",
                "Partial validity map", 
                output_dir,
                cmap=pandora_cmap().reversed())

# Step 2 : Compute at sub-pixel's level

Now it performs sub-pixel's precision.
It will use the origin coordinates as initial disparity and a range of +/- 1 pixel with sub-pixel's options.

#### Instantiate the machine

As it has been created, it only requires to re-start.

In [ ]:
pandora2d_machine.run_exit()

#### Define pipeline configuration

The differences are highlighted as comments in the dictionary.

In [ ]:
user_cfg = {
    "input": {
        "left": {
            "img": "data/left.tif",
            "nodata": "NaN",
        },
        "right": {
            "img": "data/right.tif",
        },
        # Initial disparity now read the output of the previous trigger,
        # it only search with a one-pixel range.
        "col_disparity": {"init": "outputs/origin_coordinates_pixel_level_run/disparity_map/", "range": 1},
        "row_disparity": {"init": "outputs/origin_coordinates_pixel_level_run/disparity_map/", "range": 1},
    },
    "ROI":{
        "col": {"first": 10, "last": 100},
        "row": {"first": 10, "last": 100}
    },
    "pipeline":{
        # Matching_cost method now runs with a subpixel precision
        "matching_cost" : {
            "matching_cost_method": "zncc",
            "subpix": 4,
            "window_size": 5,
            "step" : [5,3]
        },
        "disparity": {
            "disparity_method": "wta",
            "invalid_disparity": np.nan
        },
        # Add a refinement step, targets 1/(2^4) precision
        "refinement" : {
            "refinement_method": "dichotomy",
            "iterations": 4,
            "filter": {"method": "bicubic"},
        }
    },
    "output": {
        "path": "outputs/origin_coordinates_sub_pixel_level_run"
    },
}

#### Check the configuration and sequence of steps

The ROI will be erased by the previous config.
It raises a warning when run.

In [ ]:
checked_cfg = check_conf(user_cfg, pandora2d_machine)

In [ ]:
pipeline_cfg = checked_cfg['pipeline']
pprint(pipeline_cfg)

#### Prepare the machine : Only the region of interest (ROI) is read and returned

#### Run Pandora2D machine

The output will have no additional no-data on edges thanks to the included processing margins.

In [ ]:
dataset, _ = pandora2d.run_pandora2d(pandora2d_machine, checked_cfg)

Visualize output disparity map

In [ ]:
plot_two_images(dataset["row_map"].data,
                dataset["col_map"].data,
                "Row disparity map",
                "Columns disparity map", 
                output_dir,
                cmap=pandora_cmap())

In [ ]:
plot_two_images(dataset["validity"].data[..., 0],
                dataset["validity"].data[..., 1],
                "Validity map",
                "Partial validity map", 
                output_dir,
                cmap=pandora_cmap().reversed())